# New baseline on a new dataset

**Rappel des conclusions de l'EDA:**

1.  Standardisation du Format : Conversion systématique RGBA en RGB pour stabiliser les canaux d'entrée.
2.  Harmonisation de la Résolution : Redimensionnement unifié en 224x224 px. Pour la classe Oily, nous utiliserons une interpolation de type `CUBIC` pour limiter la perte de qualité lors de l'upscaling.
3.  Auto-Crop & Recentrage : Utilisation de MediaPipe Face Detection pour isoler le visage et éliminer les fonds (souvent responsables des pics de saturation à 255 dans la classe Normal).
4.  Data Augmentation Ciblée :
    * Flips horizontaux pour la diversité.
    * Variations de luminosité/contraste pour simuler différents éclairages sur le sébum.
    * Ajout de flou léger (Gaussian Blur) sur les classes de haute résolution pour neutraliser le biais de qualité face à la classe Oily.
5.  Stratégie de Modélisation : Recours au Transfer Learning (MobileNetV2) pour compenser la faible volumétrie (900 images) et capitaliser sur des extracteurs de textures pré-entraînés.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image, ImageFilter
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import random
import os
from pathlib import Path
import matplotlib.pyplot as plt

# Téléchargement du modèle de détection de visage
if not os.path.exists('detector.task'):
    print("Téléchargement du modèle MediaPipe...")
    os.system('wget -q -O detector.task https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.task')

# Config de base
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ['dry', 'normal', 'oily']
DATA_DIR = Path("data/raw/skin_images/generated_dataset") #modifier

In [ ]:
class SkinPreprocessing(Dataset):
    def __init__(self, file_list, labels, transform=None, augment=False):
        self.file_list = file_list
        self.labels = labels
        self.transform = transform
        self.augment = augment
        
        # Initialisation du détecteur MediaPipe
        base_options = python.BaseOptions(model_asset_path='detector.task')
        options = vision.FaceDetectorOptions(base_options=base_options)
        self.detector = vision.FaceDetector.create_from_options(options)

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = str(self.file_list[idx])
        label = self.labels[idx]
        
        # Étape 1 : Conversion RGB (Point 1 EDA)
        img_pil = Image.open(img_path).convert("RGB")
        
        # Étape 2 : Auto-Crop MediaPipe (Point 3 EDA)
        mp_image = mp.Image.create_from_numpy(np.array(img_pil))
        detection_result = self.detector.detect(mp_image)
        
        if detection_result.detections:
            bbox = detection_result.detections[0].bounding_box
            w, h = img_pil.size
            img_pil = img_pil.crop((max(0, bbox.origin_x), max(0, bbox.origin_y), 
                                    min(w, bbox.origin_x + bbox.width), 
                                    min(h, bbox.origin_y + bbox.height)))

        # Étape 3 : Resize BICUBIC (Point 2 EDA)
        img_pil = img_pil.resize((224, 224), resample=Image.BICUBIC)

        # Étape 4 : Flou anti-biais sur Dry et Normal (Point 4 EDA)
        if self.augment and label in [0, 1]: 
            if random.random() > 0.5:
                img_pil = img_pil.filter(ImageFilter.GaussianBlur(radius=1))

        # Étape 5 : Application de la Data Augmentation (Transform)
        if self.transform:
            img_pil = self.transform(img_pil)

        return img_pil, label

In [ ]:
# Définition des augmentations (Géométrie et Couleur)
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Récupération de tous les fichiers
all_files, all_labels = [], []
for idx, cls in enumerate(CLASSES):
    paths = list((DATA_DIR / cls).glob("*.png")) + list((DATA_DIR / cls).glob("*.jpg"))
    all_files.extend(paths)
    all_labels.extend([idx] * len(paths))

In [ ]:
# Split 70% Train / 30% Val+Test
from sklearn.model_selection import train_test_split
train_f, temp_f, train_l, temp_l = train_test_split(all_files, all_labels, test_size=0.3, stratify=all_labels, random_state=42)
val_f, test_f, val_l, test_l = train_test_split(temp_f, temp_l, test_size=0.5, stratify=temp_l, random_state=42)

In [ ]:
# Création des instances finales
train_dataset = SkinPreprocessing(train_f, train_l, transform=train_transforms, augment=True)
val_dataset = SkinPreprocessing(val_f, val_l, transform=val_transforms, augment=False)
test_dataset = SkinPreprocessing(test_f, test_l, transform=val_transforms, augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# 1. Chargement MobileNetV2
model = models.mobilenet_v2(weights='DEFAULT')
for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(nn.Dropout(0.3), nn.Linear(num_ftrs, 3))
model = model.to(DEVICE)

In [ ]:
# 2. Optimisation
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

In [ ]:
import time
import copy
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

def train_model(model, criterion, optimizer, scheduler, num_epochs=15):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Époque {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloader:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Sauvegarde de l'historique
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
                scheduler.step(epoch_loss)
                
                # Sauvegarde du meilleur modèle
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Entraînement terminé en {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Meilleure Accuracy Val: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
# --- LANCEMENT DE L'ENTRAÎNEMENT ---
model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=15)

## Comparer avec la baseline entrainée sur le dataset Kaggle